# 6 · Graph schema — declare the shape, then have Postgres hold you to it

Notebook 05 constrained *properties*. This one declares the shape of the whole
graph — which node types exist, what each carries, and which edge kinds connect
which node types — and then, as a separate and explicit step, compiles that
declaration into constraints the database enforces on every write path.

Three things are deliberately separate here, and the separation is the design:

1. **Declaring** (`define_schema`) is in-memory and instant. Nothing is validated.
2. **Enforcing** (`enforce_schema`) is DDL. `ADD CONSTRAINT` validates every
   existing row, which can fail on data that grew before the schema did.
3. **Inferring** (`infer_schema`) is an *observation* of the rows you already have.
   Adopting it is your call, in a separate line.

In [1]:
from dataclasses import dataclass
from typing import Optional

from demo_graph import connect, seed
from hopai import ConstraintViolation, Start

graph = connect("nb_06_schema")
seed(graph)
print(graph.schema, "<- nothing declared yet")

None <- nothing declared yet


## Declaring, in classes

Plain dataclasses. A field with no default is required; `Optional[X]` is optional
*and* may be JSON null. An **edge** class names its endpoints as fields annotated
with node classes — the way an ORM association object does — and everything else on
it is a property.

In [2]:
@dataclass
class Person:
    email: str                  # no default -> required
    name: str
    age: Optional[int] = None   # optional, may be null
    city: Optional[str] = None
    active: Optional[bool] = None


@dataclass
class Company:
    name: str
    founded: Optional[int] = None


@dataclass
class Friend:
    source: Person              # endpoint, not a property
    target: Person


@dataclass
class WorksAt:
    source: Person
    target: Company
    since: int                  # property


graph.define_schema(nodes=[Person, Company], edges=[Friend, WorksAt])

GraphSchema(node_types=(NodeType(name='person', properties=(Property(name='email', json_type=('string',), required=True), Property(name='name', json_type=('string',), required=True), Property(name='age', json_type=('null', 'number'), required=False), Property(name='city', json_type=('null', 'string'), required=False), Property(name='active', json_type=('boolean', 'null'), required=False))), NodeType(name='company', properties=(Property(name='name', json_type=('string',), required=True), Property(name='founded', json_type=('null', 'number'), required=False)))), edge_types=(EdgeType(kind='friend', source='person', target='person', properties=()), EdgeType(kind='works_at', source='person', target='company', properties=(Property(name='since', json_type=('number',), required=True),))))

Class names become snake_case type names — `WorksAt` → `works_at` — matching how
the Cypher examples spell kinds. The annotation mapping is one deterministic table:

```
str        -> "string"      dict / dict[...]  -> "object"
int, float -> "number"      list / list[...]  -> "array"
bool       -> "boolean"     Optional[X]       -> X's type + "null", not required
```

Anything else — a `datetime`, an `Enum`, a union that is not `Optional` — is refused
with the explicit-`Property` rewrite named, rather than coerced into a guess.

Pydantic v2 models work anywhere a dataclass does. And the same schema can be
spelled with primitives when there is no class to hand over — all three inputs
normalize to the identical canonical form:

```python
graph.define_schema(
    nodes=[NodeType("person", properties=[Property("email", "string", required=True)])],
    edges=[EdgeType("works_at", source="person", target="company")],
)
```

## Reading it back, in whichever vocabulary the consumer speaks

In [3]:
import json

print(json.dumps(graph.schema_json, indent=2)[:700], "...")

{
  "nodes": {
    "person": {
      "type": "object",
      "properties": {
        "email": {
          "type": "string"
        },
        "name": {
          "type": "string"
        },
        "age": {
          "type": [
            "null",
            "number"
          ]
        },
        "city": {
          "type": [
            "null",
            "string"
          ]
        },
        "active": {
          "type": [
            "boolean",
            "null"
          ]
        }
      },
      "required": [
        "email",
        "name"
      ]
    },
    "company": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string"
        },
         ...


`graph.schema_json` is JSON Schema vocabulary, `json.dumps`-clean — made to be
pasted into a system prompt or returned as a tool result, so a model writing
traversals knows what the properties are called before it guesses.

There are three more views of the same object: `graph.schema` (the canonical
dataclasses, and `None` until `define_schema` is called — that is the existence
check), `graph.schema_networkx` (a meta-graph of types), and
`graph.schema_pydantic` (generated models, one per type).

In [4]:
meta = graph.schema_networkx           # pip install hopai[networkx]
print(meta, "->", list(meta.edges(keys=True)))

models = graph.schema_pydantic         # pip install hopai[pydantic]
print(models)
print(models["person"](email="a@example.com", name="Ada"))

MultiDiGraph with 2 nodes and 2 edges -> [('person', 'person', 'friend'), ('person', 'company', 'works_at')]
{'person': <class 'hopai.schema.Person'>, 'company': <class 'hopai.schema.Company'>, 'friend': <class 'hopai.schema.Friend'>, 'works_at': <class 'hopai.schema.WorksAt'>}
email='a@example.com' name='Ada' age=None city=None active=None


## Enforcing

`schema_ddl()` shows the SQL first, as always. Presence and JSON type per node type
and edge kind compile to guarded CHECK constraints:

In [5]:
for statement in graph.schema_ddl()[:3]:
    print(statement, "\n")
print(f"... {len(graph.schema_ddl())} statements in total")

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_req_default_person" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR (properties ?& ARRAY['email', 'name'])) 

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_typ_default_person_email" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR jsonb_typeof(properties['email']) = 'string') 

ALTER TABLE "nodes" ADD CONSTRAINT "ck_schema_typ_default_person_name" CHECK (graph_id != 'default' OR (properties ->> 'type') IS DISTINCT FROM 'person' OR jsonb_typeof(properties['name']) = 'string') 

... 11 statements in total


The `IS DISTINCT FROM 'person'` guard is what makes each rule apply only to rows of
its own type — and vacuously true for rows carrying no `type` at all, so untyped
rows pass by construction rather than by accident.

Before running any of it on a graph that grew first, ask what would break.
`schema_violations()` is read-only and returns the whole work list, where
`ADD CONSTRAINT` would fail opaquely on the first bad row:

In [6]:
print(graph.schema_violations() or "no violations -- enforce_schema() would succeed")

no violations -- enforce_schema() would succeed


In [7]:
# Add a row that breaks the contract, then ask again.
graph.add_nodes([{"type": "person", "name": "Mallory", "age": "not a number"}])
report = graph.schema_violations()
print(bool(report))
print(report)

True
2 row(s) violate 2 schema rule(s):
  ck_schema_req_default_person (nodes): 1 row(s), e.g. id 8
  ck_schema_typ_default_person_age (nodes): 1 row(s), e.g. id 8


Two rules broken by one row: `email` is required and missing, and `age` is declared
`number` but holds a string. The report names the constraint that would be created,
how many rows fail it, and sample ids — the work list, not a first casualty.

Enforcing now fails, and names the rule the data breaks. Note the exception type:
a rejected **write** is translated into `ConstraintViolation`, but this is DDL
failing, so the driver's `IntegrityError` comes through as-is — one more reason to
ask `schema_violations()` first rather than learn about the data from a failed
migration:

In [8]:
try:
    graph.enforce_schema()
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc).splitlines()[0][:160]}")

IntegrityError: (psycopg2.errors.CheckViolation) check constraint "ck_schema_req_default_person" of relation "nodes" is violated by some row


There is no update API here yet, so the fix in this notebook is to drop the bad row
directly through SQLAlchemy — in a real system it is whatever your data-repair path
is, driven by the ids the report handed you.

In [9]:
from sqlalchemy import text

with graph.engine.begin() as conn:
    conn.execute(text("DELETE FROM nodes WHERE properties->>'name' = 'Mallory'"))

print(graph.schema_violations() or "clean")
applied = graph.enforce_schema()
print(f"{len(applied)} constraints in force, e.g. {applied[:2]}")

clean
11 constraints in force, e.g. ['ck_schema_req_default_person', 'ck_schema_typ_default_person_email']


Now every write path is validated by the server — `add_nodes`, `merge_nodes`, a
Cypher `CREATE`, or SQL from another service — and a violation surfaces as
`ConstraintViolation`:

In [10]:
for row in ({"type": "person", "name": "Nobody"},                    # no email
            {"type": "person", "name": "Ninety", "email": "n@example.com", "age": "90"}):
    try:
        graph.add_nodes([row])
        print("accepted (!)", row)
    except ConstraintViolation as exc:
        print(f"{exc.constraint}\n   {exc}")

ck_schema_req_default_person
   node rejected by constraint 'ck_schema_req_default_person' -- Failing row contains (9, default, {"name": "Nobody", "type": "person"}).
ck_schema_typ_default_person_age
   node rejected by constraint 'ck_schema_typ_default_person_age' -- Failing row contains (10, default, {"age": "90", "name": "Ninety", "type": "person", "email": "n@ex...).


`enforce_schema()` is idempotent **and reconciles**: re-running it after a schema
change drops the schema-derived constraints the current schema no longer produces,
so it converges instead of accreting. It only ever touches objects it named
(`ck_schema_*`) — your `define_constraints()` declarations are not its to drop.

## Endpoint types need a trigger, and cost something

"`works_at` connects a person to a company" cannot be a CHECK: a CHECK sees one
row, and this rule needs the endpoint *nodes*. So it is an opt-in backed by a
constraint trigger, priced per edge write — stated here rather than switched on
quietly.

In [11]:
graph.enforce_schema(endpoints=True)

# A `robot` node is fine on its own: every schema CHECK is guarded by type, so
# a type the schema never mentions passes by construction.
graph.add_nodes([{"type": "robot", "name": "R2"}])
robot = graph.traverse(Start(where={"name": "R2"})).nodes[0]["id"]
acme = graph.traverse(Start(where={"name": "Acme"})).nodes[0]["id"]

try:
    graph.add_edges([{"start_id": robot, "end_id": acme, "kind": "works_at", "since": 2024}])
except ConstraintViolation as exc:
    print(exc)

edge rejected by constraint 'ck_schema_end_default' -- works_at connects robot -> company, but the schema declares: friend: person -> person; works_at: person -> company


The message names what was written and what the schema declares. Note the limit
this has, and it is inherent rather than an oversight: the trigger validates edges
**as they are written**. Retyping a node under existing edges is not re-checked.

Calling `enforce_schema()` again without `endpoints=True` drops the trigger — the
same reconciliation as the CHECKs.

## Inferring, when the graph came first

Never declared anything and have a million rows? The schema is sitting in the data
and Postgres can compute it — a few `GROUP BY`s over JSONB.

In [12]:
chaos = graph.in_graph("chaotic")
chaos.add_nodes([
    {"type": "person", "email": "a@example.com", "age": 30},
    {"type": "person", "email": "b@example.com"},                  # no age
    {"type": "person", "email": "c@example.com", "age": "31"},     # age as a string
    {"type": "company", "name": "Acme"},
    {"name": "who knows"},                                         # no type at all
])
chaos.add_edges([{"start": {"email": "a@example.com"}, "end": {"name": "Acme"},
                  "kind": "works_at", "since": 2019}])

inferred, report = chaos.infer_schema()
print(report)

nodes: 4 typed across 2 type(s) {'company': 1, 'person': 3}, 1 untyped (outside the schema)
edges: 1 with a kind across 1 kind(s) {'works_at': 1}, 0 kindless, 0 skipped (endpoint node carries no type)
conflict: nodes/person.age observed as ['number', 'string']


The report is the honest half. A property on *every* row of its type infers
required; missing on some infers optional; an observed null infers nullable; and a
key holding both `42` and `"42"` infers the **type set** `["number", "string"]` plus
a conflict line — never a silently picked winner. Rows with no `type` cannot be
invented into a type, so they are counted and left alone.

In [13]:
for node_type in inferred.node_types:
    print(node_type.name, [(p.name, p.json_type, p.required) for p in node_type.properties])
for edge_type in inferred.edge_types:
    print(edge_type.kind, edge_type.source, "->", edge_type.target)

company [('name', ('string',), True)]
person [('age', ('number', 'string'), False), ('email', ('string',), True)]
works_at person -> company


Nothing is registered by that call. Adopting the observation as the contract is a
separate line — which is exactly why `infer_schema()` is a method and not a silent
`.schema` fallback:

```python
inferred, report = graph.infer_schema()
print(report)                             # read this first
graph.define_schema(schema=inferred)      # adopt it -- your call
graph.enforce_schema()                    # chaotic graph, now server-validated
```

Enforcing *that* schema as-is would bless `age` holding both numbers and strings.
Tightening it to `Property("age", "number")` and then running `schema_violations()`
gives you the list of rows to fix first — the loop this API is shaped for.

---

Next: [07 · Many graphs](07_many_graphs.ipynb) — thousands of isolated graphs in one
pair of tables.